# Task 2.2 Reproduction of Suboptimal SVM Solution Path

**Goal:** Implement the suboptimal solution path algorithm (Section 3) with $\epsilon_1$ and $\epsilon_2$ tolerance.

**Mechanism:** 
1. Initialize at a very small regularization value.
2. Use relaxed KKT conditions to define index sets $M$ (margin), $O$ (outside), $I$ (inside/misclassified).
3. Compute path direction $\beta$ using a linear system update.
4. Determine the step length $\Delta \theta$ that leads to the next breakpoint.
5. Update multiple points at once if they hit the relaxed boundary.

**Evaluation Metric:** Accuracy and number of breakpoints.

## Implementation

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Set seed
np.random.seed(42)

# Load data
df = pd.read_csv('data/toy_dataset.csv')
X = df[['feature1', 'feature2']].values
y = df['target'].values
n = len(y)

# Kernel matrix Q (RBF kernel as in Section 5)
def rbf_kernel(X1, X2, gamma=0.5):
    sq_dist = np.sum(X1**2, 1).reshape(-1, 1) + np.sum(X2**2, 1) - 2 * np.dot(X1, X2.T)
    return np.exp(-gamma * sq_dist)

gamma = 0.5
Q_raw = rbf_kernel(X, X, gamma)
# Add small epsilon to diagonal for stability as suggested in Section 5
Q = Q_raw * np.outer(y, y) + 1e-6 * np.eye(n)

def get_f_outputs(alpha, alpha0, Q_raw, y):
    """Calculates f(x_i) outputs."""
    # f(x) = sum alpha_j y_j K(x, x_j) + alpha0
    return np.dot(Q_raw, alpha * y) + alpha0

def solve_svm_standard(C):
    """Utility to solve standard SVM for initialization using scipy."""
    cons = ({'type': 'eq', 'fun': lambda a: np.dot(a, y)})
    bounds = [(0, C) for _ in range(n)]
    def obj(a):
        return 0.5 * np.dot(a.T, np.dot(Q, a)) - np.sum(a)
    res = minimize(obj, np.zeros(n), bounds=bounds, constraints=cons)
    alpha = res.x
    # alpha0 estimation from margin points
    margin_idx = np.where((alpha > 1e-5) & (alpha < C - 1e-5))[0]
    if len(margin_idx) > 0:
        f_vals = np.dot(Q_raw[margin_idx], alpha * y)
        alpha0 = np.mean(y[margin_idx] - f_vals)
    else:
        alpha0 = 0
    return alpha, alpha0

### Initial Solution
The code below initializes the SVM at a small $C=0.01$ to begin tracing the path.
This corresponds to Section 2.2 initialization.

In [ ]:
C_init = 0.01
alpha, alpha0 = solve_svm_standard(C_init)
print(f"Initial solution found for C={C_init}")

### Suboptimal Path Logic (Core Reproduction)
Implementing the relaxed index set selection and gradient calculations manually for the first segment.

In [ ]:
def get_index_sets(alpha, alpha0, C, e=0.01):
    """Partitions data based on relaxed KKT (Equation 6)."""
    f = get_f_outputs(alpha, alpha0, Q_raw, y)
    yf = y * f
    eps1 = e
    eps2 = e * C
    
    O = np.where((yf >= 1 - eps1) & (alpha <= eps2))[0]
    M = np.where((yf >= 1 - eps1) & (yf <= 1 + eps1) & (alpha >= -eps2) & (alpha <= C + eps2))[0]
    I_set = np.where((yf <= 1 + eps1) & (alpha >= C - eps2))[0]
    
    # Handle overlap by priority (M takes precedence to avoid singularity if possible)
    M_set = set(M)
    O_set = set(O) - M_set
    I_set = set(I_set) - M_set - O_set
    
    return list(O_set), list(M_set), list(I_set)

def calculate_gradients(M, I, y, Q, d_I):
    """Theorem 1: Calculates gradients beta and g."""
    yM = y[M]
    QM = Q[np.ix_(M, M)]
    
    # Matrix M from Theorem 1
    top = np.array([0])
    top = np.append(top, yM).reshape(1, -1)
    bottom_left = yM.reshape(-1, 1)
    bottom = np.hstack([bottom_left, QM])
    M_matrix = np.vstack([top, bottom])
    
    # Right hand side
    # In our test, we assume C grows equally for all points, so d_I is 1s
    yI = y[I]
    QM_I = Q[np.ix_(M, I)]
    rhs = -np.vstack([np.dot(yI, d_I), np.dot(QM_I, d_I)])
    
    try:
        sol = np.linalg.solve(M_matrix + 1e-9 * np.eye(len(M_matrix)), rhs)
        beta0 = sol[0][0]
        betaM = sol[1:].flatten()
    except np.linalg.LinAlgError:
        # Fallback to least squares if singular
        sol, _, _, _ = np.linalg.lstsq(M_matrix, rhs, rcond=None)
        beta0 = sol[0][0]
        betaM = sol[1:].flatten()
        
    beta = np.zeros(n)
    beta[M] = betaM
    beta[I] = d_I
    
    # g = Q * beta + y * beta0 (Equation after Theorem 1)
    # Note: the paper defines g as change of yif_i
    g = np.dot(Q, beta) + y * beta0 
    
    return beta0, beta, g

O, M, I_set = get_index_sets(alpha, alpha0, C_init)
b0, b, g = calculate_gradients(M, I_set, y, Q, np.ones(len(I_set)))
print(f"Gradients calculated. Size of M (margin set): {len(M)}")

### Step Length Calculation
The code below implements the breakpoint detection logic from Section 3.1.

In [ ]:
def get_step_length(alpha, alpha0, C, b, b0, g, O, M, I_set, e=0.01, d_path=1.0):
    f = get_f_outputs(alpha, alpha0, Q_raw, y)
    yf = y * f
    eps1 = e
    eps2 = e * C
    
    candidates = []
    
    # ThetaO (Equation page 4)
    for i in O:
        if g[i] < -1e-10:
            candidates.append((1 - eps1 - yf[i]) / g[i])
            
    # ThetaMl
    for i in M:
        if b[i] < -1e-10:
            candidates.append(-(alpha[i] + eps2) / b[i])
            
    # ThetaMu
    for i in M:
        if b[i] > d_path + 1e-10:
            candidates.append((C + eps2 - alpha[i]) / (b[i] - d_path))
            
    # ThetaI
    for i in I_set:
        if g[i] > 1e-10:
            candidates.append((1 + eps1 - yf[i]) / g[i])
            
    valid_cands = [c for c in candidates if c > 1e-12]
    return min(valid_cands) if valid_cands else 0.5 # Default move if none

delta_theta = get_step_length(alpha, alpha0, C_init, b, b0, g, O, M, I_set)
print(f"Next breakpoint step length: {delta_theta}")